# HWP / HWPX (한글) (2026년 최신 권장 사용법)

한글(HWP)은 한글과컴퓨터에서 개발한 워드프로세서로, 한국의 대표적인 문서 작성 프로그램입니다.

파일 확장자로 `.hwp`(바이너리) 와 `.hwpx`(XML 기반 개방형 포맷, KS X 6101) 를 사용하며, 기업·학교·정부 기관에서 널리 활용됩니다.

> **⚠️ 2026년 9월 기준 변경 사항**
>
> - 책에서는 LangChain 에 HWP 통합이 없어 저자가 만든 `langchain-teddynote` 의 `HWPLoader` 를 사용했습니다.
> - 현재는 LangChain 공식 문서의 Document loader 목록(Common file types)에 **`HwpHwpxLoader`** (`langchain-hwp-hwpx-loader` 패키지)가 등재되어 있습니다.
>   - 순수 Python 구현(`hwp-hwpx-parser` 기반)이라 오프라인/온프렘 환경에서 사용 가능
>   - `.hwp` 와 `.hwpx` 모두 지원, 표를 Markdown 으로 변환, 각주/미주/메모/하이퍼링크 추출
>   - `mode="single"`(문서 1개) / `mode="elements"`(본문·표·각주 등 요소 단위)
>   - 디렉토리 일괄 로딩용 `HwpHwpxDirectoryLoader` 제공
> - 아직 0.1.x 버전의 커뮤니티 패키지이므로, 중요한 파이프라인에서는 결과를 꼭 검수하세요.

In [ ]:
# 설치
# !pip install -qU langchain-hwp-hwpx-loader

In [ ]:
from langchain_hwp_hwpx import HwpHwpxLoader

# HWP Loader 객체 생성 (문서 전체를 Document 1개로)
loader = HwpHwpxLoader("./data/디지털 정부혁신 추진계획.hwp", mode="single")

# 문서 로드
docs = loader.load()

In [ ]:
# 결과 출력
print(docs[0].page_content[:1000])

In [ ]:
len(docs)

In [ ]:
print(docs[0].page_content)

metadata 에는 파일 경로 등 문서 정보가 담겨 있습니다.

In [ ]:
# 결과 출력
print(docs[0].metadata)

## 추출 옵션 지정

표 출력 형식, 이미지 표시 방식 등을 `hwp_hwpx_parser.ExtractOptions` 로 지정할 수 있습니다.

In [ ]:
from hwp_hwpx_parser import ExtractOptions, ImageMarkerStyle, TableStyle

options = ExtractOptions(
    table_style=TableStyle.MARKDOWN,  # 표를 Markdown 표로 변환
    image_marker=ImageMarkerStyle.SIMPLE,  # 이미지 위치에 간단한 표시 삽입
)

loader = HwpHwpxLoader(
    "./data/디지털 정부혁신 추진계획.hwp",
    mode="single",
    extract_options=options,
    include_tables=True,
    include_notes=True,  # 각주/미주
    include_memos=True,  # 메모
    include_hyperlinks=True,
)
docs = loader.load()
print(docs[0].page_content[:1000])

## 요소 단위 로딩 (`mode="elements"`)

본문/표/각주/미주/메모/링크/이미지를 각각 별도의 `Document` 로 반환합니다. `lazy_load()` 로 하나씩 처리할 수 있습니다.

In [ ]:
from collections import Counter

loader = HwpHwpxLoader("./data/디지털 정부혁신 추진계획.hwp", mode="elements")

element_docs = list(loader.lazy_load())
print(len(element_docs))
Counter(d.metadata["element_type"] for d in element_docs)

In [ ]:
for d in element_docs[:5]:
    print(d.metadata["element_index"], d.metadata["element_type"], "|", d.page_content[:80])

## 분할하여 RAG 에 사용

로더와 분할기를 분리해서 사용합니다. (`load_and_split` 대신 `split_documents`)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
split_docs = text_splitter.split_documents(docs)
print(len(split_docs))
split_docs[0]

## 폴더 단위 로딩

디렉토리를 재귀 탐색해 `.hwp`, `.hwpx` 파일을 순서대로 로드합니다.

In [ ]:
from langchain_hwp_hwpx import HwpHwpxDirectoryLoader

dir_loader = HwpHwpxDirectoryLoader(
    dir_path="./data",
    glob="**/*",
    recursive=True,
    mode="single",
    on_error="warn",  # 실패한 파일은 경고만 하고 건너뜀
)
hwp_docs = dir_loader.load()
print(len(hwp_docs))